# Free-Droid (Szabi) — v13 fine-tune (CSAK 8B, EGYVÁLTOZÓS: `lora_r` 8 → 16)

Vékony futtató: telepíti az Unsloth-ot, klónozza a repót, és a `training/finetune.py`-t hívja.
Minden logika a `finetune.py` + `config.py`-ban (verziókövetett).

## Mi ez a kör

**Egyetlen dolog változik: az adapter kapacitása.** A `lora_r` 8 → 16, és **az `alpha` VELE
EGYÜTT** 8 → 16. Ez utóbbi nem részletkérdés: LoRA-nál a tényleges skálázás `alpha/r`, tehát
ha csak az `r`-t emelnénk, a hatás **felére csökkenne** — az már két változtatás lenne egyben,
ellenkező előjellel. Az arány marad 1,0, csak a rang nő. A `config.py` kikommentelt `medium`
presetje pont ezért nem használható: az a `learning_rate`-et és az `epochs`-ot is mozgatja.

Minden más változatlan: `lr=5e-5`, `epochs=3`, bázismodell, rendszerprompt, dataset.

**Műszer:** az `eval_loss` görbe (VÉTÓ, nem választás — ha emelkedik, a kör bukott), plusz a
`tool_reliability.py` és a vak, bináris benchmark a v12 ellen. A `lora_r` az egyetlen olyan
knob a hátralévők közül, aminek van önálló, olcsó műszere.

## Miért NINCS 3B ebben a körben

A 2026-08-09-i mérés eldöntötte: a modell nélküli **edge-relé 23/25**, a `szabi-3b-v12 +RAG`
**11/25** ugyanazon a 25 kérdésen (12 nyerés, 0 vesztés, párosított előjelteszt **p = 0,0005**).
Nincs egyetlen kérdés sem, amit a 3B tud és a relé nem. A 3B fine-tune ezért **kimarad**; a
végső döntést a Pi 5-ön végzett élő teszt adja meg a jövő héten.

Ez PoC-eredményként is kimondható a színpadon: **8B alatt nem éri meg LLM-et futtatni** erre a
feladatra — a szabályalapú relé mérhetően jobb.

## ⚠️ A `scan_wifi` regresszió NEM ebben a körben dől el — és tudni kell, miért

A v12 elbukja a `tc_04`-et („Szabi, mit látsz a hálózaton?"), amit a v8 és a v11 megold.
Kézenfekvő lett volna ide tenni a javítást, de a dataset-ellenőrzés megcáfolta a feltevést:

**a kérdés SZÓ SZERINT benne van a tanítóadatban** (`dataset/freedroid_full.json`, a 18
`scan_wifi`-példa egyike), és a tool-példák aránya **17,6%** — tehát ez nem lefedettségi hiány.
A v12 egy olyan példát felejtett el, amit betűre látott. A valószínű ok a v12 saját újítása:
a 94 RAG-grounding példa azt tanítja, hogy „kérdés → fogalmazz a tudásodból", a
„Mit látsz a hálózaton?" pedig **kérdés alakú parancs** — a két viselkedés versenyez.

Ha ez így van, **több `scan_wifi`-példa nem javítja meg**, mert nem a mennyiség hiányzik. A
helyes válaszok sorrendben: (1) az orchestrátor a parancs-szándékot a parserhez irányítja a
modell előtt — ezt a relé-mérés már bizonyította, ott a `tool_calling` **5/5**; (2) ha marad
maradék hiba, **kontrasztív** példapár kell (azonos kérdés-alak, egyszer tool-válasz, egyszer
RAG-válasz), nem több azonos példa. Egyik sem `lora_r`-kérdés, tehát nem ebbe a körbe való.


## Unsloth telepítése

In [ ]:
# 1. Unsloth telepítése (hivatalos Colab-installer — illeszti a torch/bnb/triton verziókat).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes


## Repo klónozása + guard

In [ ]:
# 2. Repo a kívánt ágról, majd be a training/-be.
# PR-staging alatt állítsd a feature-ágra; merge után hagyd "main"-en.
import os, shutil

BRANCH = "main"

# Újrafuttatható: előbb vissza /content-be és el a korábbi klónnal. Enélkül a cella
# második futtatása a training/ ALÁ klónozna (a %cd megmarad a session-ben).
%cd /content
if os.path.exists("free-droid"):
    shutil.rmtree("free-droid")
!git clone --depth 1 -b {BRANCH} https://github.com/pits2022/free-droid.git
%cd free-droid/training

# A guard a repóban él (training/verify_training_setup.py). Ez a kör NEM változtat
# dataseten, tehát a guard dolga itt annyi, hogy a megszokott korpuszt találja —
# ha régi ágról klónoztál, az azonnal kiderül, nem 4 óra múlva.
#
# subprocess + assert, NEM `!python`: a `!parancs` nem-nulla kilépési kódja Jupyterben
# nem állítja meg a notebookot, tehát a guard csak figyelmeztetne. Az assert elhasal,
# és a "Run all" is megáll rajta.
import subprocess, sys

rc = subprocess.run([sys.executable, "verify_training_setup.py"]).returncode
assert rc == 0, "A pre-flight guard BUKOTT — ne indítsd a tanítást (lásd a kimenetet fent)."


## Cloud modell — Llama 3.1 8B (a fő demó-agy)

Az `--lora-r 16 --lora-alpha 16` a preset FÖLÉ megy: a `finetune.py` minden
TrainConfig-mezőre generál CLI-kapcsolót, tehát a `gentle` többi értéke (`lr=5e-5`,
`dropout=0.05`) változatlan marad. A futás ~4-5 óra egy T4-en.


In [ ]:
!python finetune.py --variant llama8b --preset gentle --epochs 3 \
    --lora-r 16 --lora-alpha 16 --tag v13

## Next

- **Kimenet:** `training/outputs/llama3.1-8b-v13/` (a klónon KÍVÜL, lásd a 2. cellát).
  ⚠️ **A `lora-adapter*` mappát IS töltsd le**, ne csak a GGUF-ot. A HF Space az ADAPTERT
  tölti be, GGUF-fal nem lehet átállítani. A `checkpoints/` is jöjjön le.

- ⚠️ **AZ UNSLOTH ÁLTAL GENERÁLT `Modelfile`-t NE HASZNÁLD** — nincs benne SYSTEM, és
  `temperature 1.5`-öt hoz. Mindig a `make_modelfile.py`, az a `config.py`-ból veszi a
  variánshoz tartozó promptot.

- ⚠️ **A GGUF nem ott van, ahol az export-mappa neve mutatja**: a megadott mappába a
  16-bites merge kerül, a GGUF egy `_gguf` utótagú testvérmappába.

```
python make_modelfile.py --variant llama8b \
    outputs/llama3.1-8b-v13/gguf-q4_k_m_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf
cd outputs/llama3.1-8b-v13/gguf-q4_k_m_gguf && ollama create szabi-8b-v13 -f Modelfile_<név>
```

### 1. lépés — LOSS-VÉTÓ (nem választás)

Az `eval_loss` mindhárom epochnál ereszkedjen. Ha emelkedik, a nagyobb rang túltanulást hozott,
és a kör bukott — ilyenkor nem a v13-at finomítjuk, hanem visszalépünk `r=8`-ra. Viszonyítás:
a v12 8B-je 1,493 / 1,431 / 1,429 volt. **A loss önmagában NEM minőség** (a repó régi
tanulsága), tehát ereszkedő görbénél sem dönt — csak kizár.

### 2. lépés — a két ROMLÁS-ŐR, mielőtt bármi mást mérnél

```
python tool_reliability.py --models szabi-8b-v12 szabi-8b-v13 --repeat 3
python tech_retrieval_probe.py        # 18/20 — ezt a RAG adja, a modelltől független
```

A nagyobb rang a tool-grammatikát ronthatja el először, mert az a legszűkebb, legformálisabb
minta a korpuszban. Ha a `tool_reliability` esik, a v13 nem jó, akármit mond a benchmark.

### 3. lépés — vak, bináris aréna a v12 ellen, HORGONNYAL

```
python -m freedroid.rag.corpus       # MINDIG, mielőtt --rag-gal mérsz
python run_benchmark.py --models szabi-8b-v12 szabi-8b-v13 --rag --json-out \
    --anchor benchmark_raw_2026-08-09.json --anchor-column "szabi-8b-v12 +RAG"
# kézi pontozás a .md-ben (0/1 + ok-címke), majd:
python run_benchmark.py --decode benchmark_eredmeny_<dátum>.md \
    --key benchmark_kulcs_<dátum>.json --baseline benchmark_pontok_2026-08-09.json
```

**A horgony ebben a körben kötelező**, és most már tudjuk, miért: 2026-08-09-én a
`szabi-8b-v12 +RAG` egyik nap 84%-ot, másnap 96%-ot ért el **ugyanazzal a konfiggal**, miközben
a horgony 5/5-öt egyezett. Vagyis a pontozó NEM mozdult — a különbség **mintavételi szórás**,
kb. ±3 kérdés 25-ből. Ezért egyetlen futás különbsége nem döntés: a horgony választja szét,
hogy a modell változott-e vagy a mérés zajos.

### 4. lépés — red team (a demó előtt kötelező)

```
python run_benchmark.py --models szabi-8b-v13 --benchmark-file red_team.json \
    --rag --rag-dims halluc_absztencio
```

### Amit ez a kör NEM old meg

- **`scan_wifi` / `tc_04`** — a fejlécben leírt okból nem `lora_r`-kérdés. A sorrend:
  orchestrátor-routing (a relé-mérésen `tool_calling` 5/5), és csak utána kontrasztív adat.
- **Koherencia-hossz a 8B-n** — v10→v11→v12 alatt 25 → 22 → 20 szó, miközben a v11 100–106
  szavas példákat tanított. **Két kör bukott ugyanezen**, tehát előbb diagnózis kell arról,
  miért nem transzferál a 8B-re, amikor a 3B-re igen (33 → 46). Nem adagolási kérdés.
- A benchmark **8 kérdése szó szerint a tanítóadatban** van; 24 scaffold-szennyezett példa
  (index 476–599); 45 példában maradt hibás „Teremtő" megszólítás.
